In [ ]:
import random
import json
import os
from datetime import datetime

HISTORY_FILE = "tictactoe_history.json"

# Function to print the game board
def print_board(board):
    for row in board:
        print(" | ".join(row))
        print("-" * 9)

# Function to check if a player has won and return winning cells
def check_winner(board, player):
    # Rows
    for i, row in enumerate(board):
        if all(cell == player for cell in row):
            return [(i, j) for j in range(3)]
    # Columns
    for col in range(3):
        if all(board[row][col] == player for row in range(3)):
            return [(row, col) for row in range(3)]
    # Diagonals
    if all(board[i][i] == player for i in range(3)):
        return [(i, i) for i in range(3)]
    if all(board[i][2 - i] == player for i in range(3)):
        return [(i, 2 - i) for i in range(3)]
    return None

# Function to save game history
def save_history(winner, final_board):
    history = []
    if os.path.exists(HISTORY_FILE):
        with open(HISTORY_FILE, 'r') as file:
            history = json.load(file)
    history.append({
        "timestamp": str(datetime.now()),
        "winner": winner,
        "board": final_board
    })
    with open(HISTORY_FILE, 'w') as file:
        json.dump(history, file, indent=2)

# Basic computer move (random empty cell)
def computer_move(board):
    empty_cells = [(r, c) for r in range(3) for c in range(3) if board[r][c] == ' ']
    return random.choice(empty_cells)

# Ask if playing against computer
mode = input("Play vs computer? (yes/no): ").lower()
vs_computer = mode == "yes"

# Initialize the game board
board = [[' ' for _ in range(3)] for _ in range(3)]
current_player = 'X'

for turn in range(9):
    print_board(board)

    # Get player move
    if vs_computer and current_player == 'O':
        row, col = computer_move(board)
        print(f"Computer chose: {row} {col}")
    else:
        try:
            row, col = map(int, input(f"Player {current_player}, enter row and column (0-2): ").split())
            if row not in range(3) or col not in range(3):
                raise ValueError
        except ValueError:
            print("Invalid input! Please enter numbers 0, 1, or 2.")
            continue

    # Check and apply move
    if board[row][col] == ' ':
        board[row][col] = current_player
        win_cells = check_winner(board, current_player)

        if win_cells:
            # Highlight the winning move
            for r, c in win_cells:
                board[r][c] = f"*{board[r][c]}*"
            print_board(board)
            print(f"Player {current_player} wins!")
            save_history(current_player, board)
            break

        # Switch player
        current_player = 'O' if current_player == 'X' else 'X'
    else:
        print("Invalid move! Cell already taken. Try again.")
else:
    print_board(board)
    print("It's a draw!")
    save_history("Draw", board)

# Optionally display past results
see_history = input("View game history? (yes/no): ").lower()
if see_history == "yes" and os.path.exists(HISTORY_FILE):
    with open(HISTORY_FILE, 'r') as file:
        history = json.load(file)
        print("\nRecent Games:")
        for game in history[-5:]:
            print(f"🕒 {game['timestamp']} - Winner: {game['winner']}")
            for row in game["board"]:
                print(" | ".join(row))
            print("-" * 9)